In [98]:
from pymongo import MongoClient
from bson.json_util import dumps 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from time import sleep
import json
import os
from datetime import datetime

In [55]:

client = MongoClient('mongodb://localhost:27017/')
client.drop_database('simplize')
db = client['simplize']
driver = webdriver.Chrome()

url = 'https://simplize.vn/co-phieu/nganh/tai-chinh'
driver.get(url)
sleep(2)
linh_vuc = driver.find_elements(By.XPATH, "//div[contains(@class,'simplize-row css-pmt33i')]")
collection_list = []

for chay_linh_vuc in linh_vuc:
    data = chay_linh_vuc.text
    collection_name =data
    collection = db[collection_name]
    collection.insert_one({"linhvuc": data})
    collection_list.append(data)

print(collection_list)


['Tài chính ngân hàng', 'Chứng khoán và Ngân hàng đầu tư', 'Bảo hiểm']


In [56]:
len(linh_vuc)

3

In [57]:
if linh_vuc:
    linh_vuc[0].click()  
sleep(2)

In [58]:
# Lấy toàn bộ thông tin theo XPath
cac_ma = driver.find_elements(By.XPATH, "//div[contains(@class,'css-70qvj9')]")
ma_list = []  # Tạo list để chứa các thông tin vừa lấy được

for ma in cac_ma:
    ma_list.append(ma.text)  # Thêm thông tin vào list



In [59]:
# Lấy toàn bộ thông tin theo XPath
cac_ma = driver.find_elements(By.XPATH, "//div[contains(@class,'css-70qvj9')]")
ma_list = []  # Tạo list để chứa các thông tin vừa lấy được

for ma in cac_ma:
    ma_list.append(ma.text)  # Thêm thông tin vào list



In [60]:
ma_list

['VCB',
 'BID',
 'CTG',
 'TCB',
 'VPB',
 'MBB',
 'ACB',
 'LPB',
 'HDB',
 'STB',
 'VIB',
 'SSB',
 'TPB',
 'SHB',
 'EIB',
 'MSB',
 'OCB',
 'NAB',
 'BAB',
 'EVF',
 'ABB',
 'PGB',
 'BVB',
 'VBB',
 'VAB',
 'NVB',
 'KLB',
 'SGB',
 'TIN']

# cắt bớt list để chạy nhanh :))

In [61]:
# to_remove=('CTG', 'TCB', 'VPB', 'MBB', 'ACB', 'LPB', 'HDB', 'STB', 'VIB', 'SSB', 'TPB', 'SHB', 'EIB', 'MSB', 'OCB', 'NAB', 'BAB', 'EVF', 'ABB', 'PGB', 'BVB', 'VBB', 'VAB', 'NVB', 'KLB', 'SGB')
# ma_list = [x for x in ma_list if x not in to_remove]
# print(ma_list)


# bỏ đoạn trên để lấy full list

In [62]:
# Truy cập vào từng liên kết theo định dạng
for ma in ma_list:
    lich_su_gia_url = f"https://simplize.vn/co-phieu/{ma}/lich-su-gia"  # Tạo URL từ từng giá trị trong ma_list
    driver.get(lich_su_gia_url)  # Truy cập vào từng liên kết
    sleep(2)
    # Lấy toàn bộ thông tin giá
    ls_gia = driver.find_elements(By.XPATH, "//tr[contains(@class,'simplize-table-row simplize-table-row-level-0')]")
    ls_gia_list = []  # Tạo list để chứa thông tin giá

    for row in ls_gia:
        row_data = {
            "date": row.find_element(By.XPATH, ".//td[1]").text,              # Ngày
            "opening_price": row.find_element(By.XPATH, ".//td[2]").text,     # Giá mở cửa
            "highest_price": row.find_element(By.XPATH, ".//td[3]").text,     # Giá cao nhất
            "lowest_price": row.find_element(By.XPATH, ".//td[4]").text,      # Giá thấp nhất
            "closing_price": row.find_element(By.XPATH, ".//td[5]").text,     # Giá đóng cửa
            "price_change": row.find_element(By.XPATH, ".//td[6]").text,       # Thay đổi giá
            "percent_change": row.find_element(By.XPATH, ".//td[7]").text,     # % thay đổi
            "volume": row.find_element(By.XPATH, ".//td[8]").text,             # Khối lượng
        }
        ls_gia_list.append(row_data)  # Thêm dữ liệu của hàng vào danh sách
    with open(f"{ma}_lich_su_gia.json", "w", encoding='utf-8') as json_file:
        json.dump(ls_gia_list, json_file, ensure_ascii=False, indent=4)  # Lưu danh sách vào file JSON
    # Trở về trang trước để nhấp vào mã tiếp theo
    driver.back()

collection_name = collection_list[0]  # Chọn collection đầu tiên
collection = db[collection_name]  # Truy cập collection đầu tiên

for ma in ma_list:
    json_file_path = f"{ma}_lich_su_gia.json"
    
    # Kiểm tra xem file có tồn tại không
    if os.path.exists(json_file_path):
        with open(json_file_path, "r", encoding='utf-8') as json_file:
            ls_gia_list = json.load(json_file)  # Đọc dữ liệu từ file JSON

            # Cập nhật object của mã cổ phiếu
            collection.update_one(
                {"linhvuc": collection_name},  # Tìm đối tượng có giá trị là "Tài chính ngân hàng"
                {"$set": {f"lich_su_gia.{ma}": ls_gia_list}},  # Thêm lịch sử giá vào trường con theo mã cổ phiếu
                upsert=True  # Tạo mới nếu không tồn tại
            )

In [63]:
driver.quit()

In [64]:
# Kết nối tới MongoDB
client = MongoClient("mongodb://localhost:27017/")
db = client["simplize"]
cau1 = db.list_collection_names()
print(cau1)

['Tài chính ngân hàng', 'Bảo hiểm', 'Chứng khoán và Ngân hàng đầu tư']


In [65]:
#1 in ra toàn bộ

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Lấy danh sách tất cả các collection trong database
collections = db.list_collection_names()

# Duyệt qua từng collection và lấy toàn bộ dữ liệu
all_data = {}
for collection_name in collections:
    collection = db[collection_name]
    data = list(collection.find())  # Truy vấn toàn bộ dữ liệu
    all_data[collection_name] = data  # Lưu dữ liệu vào dict với key là tên collection

# In dữ liệu ra màn hình hoặc lưu vào file JSON
for collection_name, data in all_data.items():
    print(f"Dữ liệu trong collection '{collection_name}':")
    print(dumps(data, ensure_ascii=False, indent=4))  # Sử dụng bson.json_util.dumps

# Hoặc lưu toàn bộ dữ liệu vào file JSON
with open("all_data.json", "w", encoding='utf-8') as f:
    f.write(dumps(all_data, ensure_ascii=False, indent=4))


Dữ liệu trong collection 'Tài chính ngân hàng':
[
    {
        "_id": {
            "$oid": "6720ebd6a6bae9f6d58c5dbd"
        },
        "linhvuc": "Tài chính ngân hàng",
        "lich_su_gia": {
            "VCB": [
                {
                    "date": "29/10/2024",
                    "opening_price": "92,100",
                    "highest_price": "92,600",
                    "lowest_price": "92,000",
                    "closing_price": "92,000",
                    "price_change": "-",
                    "percent_change": "-",
                    "volume": "1,297,600"
                },
                {
                    "date": "28/10/2024",
                    "opening_price": "91,500",
                    "highest_price": "92,300",
                    "lowest_price": "91,500",
                    "closing_price": "92,000",
                    "price_change": "+200",
                    "percent_change": "0.22%",
                    "volume": "1,105,800"
         

In [72]:
#2 in ra tên của tất cả các mã trong tài chính ngân hàng
# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  # Thay bằng tên chính xác của collection
collection = db[collection_name]

# Tập hợp để lưu trữ các mã cổ phiếu
ma_cophieu_set = set()  # Sử dụng set để loại bỏ mã trùng lặp

# Duyệt qua từng tài liệu trong collection
for document in collection.find({}, {"lich_su_gia": 1}):
    lich_su_gia = document.get("lich_su_gia", {})
    ma_cophieu_set.update(lich_su_gia.keys())  # Thêm tất cả mã cổ phiếu vào set

# In ra danh sách mã cổ phiếu
print("Danh sách các mã cổ phiếu:")
for ma in ma_cophieu_set:
    print(ma)

Danh sách các mã cổ phiếu:
STB
ACB
TCB
LPB
ABB
EIB
NAB
KLB
BID
CTG
MBB
VIB
BVB
OCB
NVB
HDB
MSB
BAB
EVF
TPB
VAB
VCB
PGB
SSB
SHB
TIN
VPB
VBB
SGB


In [66]:
#3 lấy dữ liệu của giá cổ phiếu của các mã trong Tài chính ngân hàng
# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Truy vấn dữ liệu từ collection "Bảo Hiểm"
collection = db['Tài chính ngân hàng']  # Thay bằng tên chính xác của collection
cursor = collection.find({}, {"_id": 0, "lich_su_gia": 1})  # Lấy trường "lich_su_gia" của tất cả các mã

# In dữ liệu ra màn hình
for document in cursor:
    print(dumps(document, ensure_ascii=False, indent=4))


{
    "lich_su_gia": {
        "VCB": [
            {
                "date": "29/10/2024",
                "opening_price": "92,100",
                "highest_price": "92,600",
                "lowest_price": "92,000",
                "closing_price": "92,000",
                "price_change": "-",
                "percent_change": "-",
                "volume": "1,297,600"
            },
            {
                "date": "28/10/2024",
                "opening_price": "91,500",
                "highest_price": "92,300",
                "lowest_price": "91,500",
                "closing_price": "92,000",
                "price_change": "+200",
                "percent_change": "0.22%",
                "volume": "1,105,800"
            },
            {
                "date": "25/10/2024",
                "opening_price": "92,100",
                "highest_price": "92,300",
                "lowest_price": "91,700",
                "closing_price": "91,800",
                "price_cha

In [67]:
#4 đếm số lượng data có trong tài chính ngân hàng

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Truy vấn collection "Tài Chính Ngân Hàng"
collection_name = "Tài chính ngân hàng"
collection = db[collection_name]

# Tổng số mã cổ phiếu và tổng số phần tử trong các mảng lịch sử giá
num_ma = 0  # Biến đếm số mã cổ phiếu
total_entries = 0  # Biến đếm tổng số phần tử trong các mảng lịch sử giá

# Duyệt qua từng tài liệu trong collection
for document in collection.find({}, {"_id": 0, "lich_su_gia": 1}):
    lich_su_gia = document.get("lich_su_gia", {})
    num_ma += len(lich_su_gia)  # Đếm số mã cổ phiếu
    for ma, gia_data in lich_su_gia.items():
        total_entries += len(gia_data)  # Đếm tổng số phần tử trong các mảng lịch sử giá của từng mã

print(f"Số lượng mã cổ phiếu trong lịch sử giá: {num_ma}")
print(f"Tổng số phần tử trong các mảng lịch sử giá của các mã: {total_entries}")


Số lượng mã cổ phiếu trong lịch sử giá: 29
Tổng số phần tử trong các mảng lịch sử giá của các mã: 870


In [68]:
#5 đếm tổng volume của một mã bất kì 
from pymongo import MongoClient

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  # Thay bằng tên collection chính xác
collection = db[collection_name]

# Tên mã cổ phiếu cần tính tổng volume
ma_cophieu = "VCB"

# Tổng volume của mã cổ phiếu
total_volume = 0

# Truy vấn tài liệu chứa mã cổ phiếu "VCB" trong "lich_su_gia"
document = collection.find_one({f"lich_su_gia.{ma_cophieu}": {"$exists": True}}, {"lich_su_gia": 1})

if document:
    lich_su_gia = document["lich_su_gia"].get(ma_cophieu, [])
    for record in lich_su_gia:
        # Chuyển đổi volume sang dạng số và cộng vào total_volume
        total_volume += int(record["volume"].replace(",", ""))  # Loại bỏ dấu phẩy nếu có trong volume
    print(f"Tổng volume của mã {ma_cophieu}: {total_volume}")
else:
    print(f"Không tìm thấy mã cổ phiếu {ma_cophieu} trong lịch sử giá.")


Tổng volume của mã VCB: 38902800


In [75]:
#6 lấy giá đóng cửa cao nhất trong một mã bất kì

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  
collection = db[collection_name]

# Tên mã cổ phiếu cần tìm
ma_cophieu = "VCB"  # Thay bằng mã cổ phiếu muốn kiểm tra

# Truy vấn tài liệu chứa mã cổ phiếu trong "lich_su_gia"
document = collection.find_one({f"lich_su_gia.{ma_cophieu}": {"$exists": True}}, {"lich_su_gia": 1})

if document:
    lich_su_gia = document["lich_su_gia"].get(ma_cophieu, [])
    
    # Tìm giá đóng cửa lớn nhất
    max_closing_price = float('-inf')  # Khởi tạo với giá trị âm vô cùng
    for record in lich_su_gia:
        closing_price = float(record["closing_price"].replace(",", ""))  # Chuyển đổi sang số
        if closing_price > max_closing_price:
            max_closing_price = closing_price  # Cập nhật giá đóng cửa lớn nhất
            
    if max_closing_price != float('-inf'):
        print(f"Giá đóng cửa lớn nhất của mã {ma_cophieu}: {max_closing_price}")
    else:
        print(f"Không có dữ liệu giá đóng cửa cho mã {ma_cophieu}.")
else:
    print(f"Không tìm thấy mã cổ phiếu {ma_cophieu} trong lịch sử giá.")


Giá đóng cửa lớn nhất của mã VCB: 92800.0


In [96]:
#7 tìm giá mở cửa lớn nhất trong tất cả các mã(mã bị lỗi khi thay thành giá mở cửa do có rỗng)

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng" 
collection = db[collection_name]

# Khởi tạo biến để lưu giá mở cửa lớn nhất
max_opening_price = float('-inf')  # Khởi tạo với giá trị âm vô cùng
max_opening_stock = None  # Biến lưu tên mã cổ phiếu tương ứng

# Duyệt qua từng tài liệu trong collection
for document in collection.find({}, {"lich_su_gia": 1}):
    lich_su_gia = document.get("lich_su_gia", {})
    for ma_cophieu, records in lich_su_gia.items():
        for record in records:
            opening_price = float(record["volume"].replace(",", ""))  # Chuyển đổi sang số
            if opening_price > max_opening_price:
                max_opening_price = opening_price  # Cập nhật giá mở cửa lớn nhất
                max_opening_stock = ma_cophieu  # Lưu mã cổ phiếu tương ứng

if max_opening_stock:
    print(f"Khối lượng lớn nhất: {max_opening_price} của mã cổ phiếu: {max_opening_stock}")
else:
    print("Không có dữ liệu giá mở cửa.")



Khối lượng lớn nhất: 62864300.0 của mã cổ phiếu: VPB


In [90]:
#7.1 lấy giá mở của lớn nhất trong tất cả các mã

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  # Thay bằng tên chính xác của collection
collection = db[collection_name]

# Khởi tạo biến để lưu giá mở cửa lớn nhất
max_opening_price = float('-inf')  # Khởi tạo với giá trị âm vô cùng
max_opening_stock = None  # Biến lưu tên mã cổ phiếu tương ứng

# Duyệt qua từng tài liệu trong collection
for document in collection.find({}, {"lich_su_gia": 1}):
    lich_su_gia = document.get("lich_su_gia", {})
    for ma_cophieu, records in lich_su_gia.items():
        for record in records:
            opening_price_str = record.get("opening_price", "").replace(",", "")  # Lấy giá mở cửa, thay thế dấu phẩy
            if opening_price_str:  # Kiểm tra xem giá mở cửa không phải là chuỗi rỗng
                try:
                    opening_price = float(opening_price_str)  # Chuyển đổi sang số
                    if opening_price > max_opening_price:
                        max_opening_price = opening_price  # Cập nhật giá mở cửa lớn nhất
                        max_opening_stock = ma_cophieu  # Lưu mã cổ phiếu tương ứng
                except ValueError:
                    print(f"Có lỗi khi chuyển đổi giá mở cửa cho mã {ma_cophieu}: {opening_price_str}")

if max_opening_stock:
    print(f"Giá mở cửa lớn nhất: {max_opening_price} của mã cổ phiếu: {max_opening_stock}")
else:
    print("Không có dữ liệu giá mở cửa.")


Giá mở cửa lớn nhất: 93900.0 của mã cổ phiếu: VCB


In [109]:
#8 lấy % thay đổi nhỏ nhất của một mã bất kì

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  
collection = db[collection_name]

# Chỉ định mã cổ phiếu mà bạn muốn tìm
ma_cophieu = "VCB"  # Thay thế bằng mã cổ phiếu bạn muốn tìm

# Khởi tạo giá trị percent_change nhỏ nhất
min_percent_change = float('inf')
min_record = None

# Lấy lịch sử giá cho mã cổ phiếu
lich_su_gia = collection.find_one({"linhvuc": collection_name}, {"lich_su_gia": 1})

if lich_su_gia and ma_cophieu in lich_su_gia["lich_su_gia"]:
    records = lich_su_gia["lich_su_gia"][ma_cophieu]
    
    # Tìm percent_change nhỏ nhất
    for record in records:
        percent_change_str = record.get("percent_change", "")
        
        # Chỉ chuyển đổi nếu percent_change_str là một số hợp lệ
        if percent_change_str and percent_change_str not in ["-", ""]:
            try:
                # Chuyển đổi percent_change thành số
                percent_change = float(percent_change_str.replace(",", "").replace("%", ""))
                if percent_change < min_percent_change:
                    min_percent_change = percent_change
                    min_record = record
            except ValueError:
                # Bỏ qua nếu không thể chuyển đổi
                continue

if min_record:
    print(f"Percent change nhỏ nhất cho mã {ma_cophieu} là {min_percent_change}%, thông tin: {min_record}")
else:
    print(f"Không tìm thấy dữ liệu cho mã cổ phiếu {ma_cophieu}.")



Percent change nhỏ nhất cho mã VCB là -0.98%, thông tin: {'date': '20/09/2024', 'opening_price': '91,800', 'highest_price': '92,000', 'lowest_price': '90,600', 'closing_price': '90,600', 'price_change': '-900', 'percent_change': '-0.98%', 'volume': '2,180,400'}


In [100]:
#9 in ra sự chêch lệch khối lượng giữa 2 data có ngày gần nhất trong một cỗ phiếu


# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  
collection = db[collection_name]

# Mã cổ phiếu bạn muốn truy vấn
ma_cophieu = "TPB"  # Thay bằng mã cổ phiếu cần tìm

# Lấy lịch sử giá của cổ phiếu
document = collection.find_one({}, {"lich_su_gia": 1})
lich_su_gia = document.get("lich_su_gia", {}).get(ma_cophieu, [])

# Sắp xếp theo ngày (giả định rằng ngày được lưu trữ dưới dạng chuỗi)
lich_su_gia.sort(key=lambda x: datetime.strptime(x["date"], "%d/%m/%Y"), reverse=True)

# Lấy 2 bản ghi gần nhất
if len(lich_su_gia) >= 2:
    recent_data = lich_su_gia[:2]
    volume_difference = int(recent_data[0]["volume"].replace(",", "")) - int(recent_data[1]["volume"].replace(",", ""))
    print(f"Sự chênh lệch khối lượng giữa 2 ngày gần nhất ({recent_data[0]['date']} và {recent_data[1]['date']}): {volume_difference}")
else:
    print("Không đủ dữ liệu để tính toán sự chênh lệch khối lượng.")


Sự chênh lệch khối lượng giữa 2 ngày gần nhất (29/10/2024 và 28/10/2024): 1842800


In [114]:
#10 in ra sự chêch lệch khối lượng giữa 2 data có ngày bất kì trong một cỗ phiếu

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  
collection = db[collection_name]

# Chỉ định mã cổ phiếu mà bạn muốn tìm
ma_cophieu = "VCB"  # Thay thế bằng mã cổ phiếu bạn muốn tìm

# Lấy lịch sử giá cho mã cổ phiếu
lich_su_gia = collection.find_one({"linhvuc": collection_name}, {"lich_su_gia": 1})

if lich_su_gia and ma_cophieu in lich_su_gia["lich_su_gia"]:
    records = lich_su_gia["lich_su_gia"][ma_cophieu]
    
    # Giả sử bạn đã biết ngày của 2 bản ghi mà bạn muốn so sánh
    date1 = "10/10/2024"  # Thay đổi ngày này theo ý bạn
    date2 = "25/10/2024"  # Thay đổi ngày này theo ý bạn

    volume1 = None
    volume2 = None

    # Tìm kiếm khối lượng cho 2 ngày
    for record in records:
        record_date = record["date"]
        if record_date == date1:
            volume1 = int(record["volume"].replace(",", ""))  # Chuyển đổi khối lượng thành số
        elif record_date == date2:
            volume2 = int(record["volume"].replace(",", ""))  # Chuyển đổi khối lượng thành số

    # Tính và in ra sự chênh lệch khối lượng
    if volume1 is not None and volume2 is not None:
        difference = abs(volume1 - volume2)  # Tính sự chênh lệch
        print(f"Sự chênh lệch khối lượng giữa {date1} và {date2} cho mã {ma_cophieu} là: {difference}")
    else:
        print("Không tìm thấy dữ liệu cho một trong hai ngày đã chỉ định.")
else:
    print(f"Không tìm thấy dữ liệu cho mã cổ phiếu {ma_cophieu}.")


Sự chênh lệch khối lượng giữa 10/10/2024 và 25/10/2024 cho mã VCB là: 328900


In [101]:
#11 lấy khối lượng lớn nhất cho hai mã bất kì và tính sự chêch lẹch khối lượng

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  
collection = db[collection_name]

# Mã cổ phiếu bạn muốn truy vấn
ma_cophieu_1 = "VCB"  # Mã cổ phiếu thứ nhất
ma_cophieu_2 = "BID"  # Mã cổ phiếu thứ hai

# Hàm để lấy khối lượng cao nhất
def get_highest_volume(ma_cophieu):
    document = collection.find_one({}, {"lich_su_gia": 1})
    lich_su_gia = document.get("lich_su_gia", {}).get(ma_cophieu, [])

    # Chuyển đổi khối lượng sang số và tìm khối lượng cao nhất
    if lich_su_gia:
        highest_volume = max(int(record["volume"].replace(",", "")) for record in lich_su_gia)
        return highest_volume
    return None

# Lấy khối lượng cao nhất cho hai mã cổ phiếu
highest_volume_1 = get_highest_volume(ma_cophieu_1)
highest_volume_2 = get_highest_volume(ma_cophieu_2)

if highest_volume_1 is not None and highest_volume_2 is not None:
    print(f"Khối lượng cao nhất cho mã {ma_cophieu_1}: {highest_volume_1}")
    print(f"Khối lượng cao nhất cho mã {ma_cophieu_2}: {highest_volume_2}")

    # So sánh khối lượng
    if highest_volume_1 > highest_volume_2:
        print(f"Khối lượng cao nhất của {ma_cophieu_1} cao hơn {ma_cophieu_2} với sự chênh lệch là {highest_volume_1 - highest_volume_2}.")
    elif highest_volume_1 < highest_volume_2:
        print(f"Khối lượng cao nhất của {ma_cophieu_2} cao hơn {ma_cophieu_1} với sự chênh lệch là {highest_volume_2 - highest_volume_1}.")
    else:
        print(f"Khối lượng cao nhất của hai mã {ma_cophieu_1} và {ma_cophieu_2} bằng nhau.")
else:
    print("Không đủ dữ liệu để tính toán.")


Khối lượng cao nhất cho mã VCB: 2602400
Khối lượng cao nhất cho mã BID: 6928400
Khối lượng cao nhất của BID cao hơn VCB với sự chênh lệch là 4326000.


In [107]:
#12 xuất tên các mã có dữ liệu từ ngày nào tới ngày nào

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  
collection = db[collection_name]

# Ngày bắt đầu và ngày kết thúc
start_date = datetime.strptime("2/10/2024", "%d/%m/%Y")  # Ngày bắt đầu
end_date = datetime.strptime("2/10/2024", "%d/%m/%Y")    # Ngày kết thúc

# Lưu trữ các mã có dữ liệu trong khoảng thời gian đã cho
ma_cophieu_with_data = []

# Lấy tất cả mã cổ phiếu
documents = collection.find({}, {"lich_su_gia": 1})

for document in documents:
    lich_su_gia = document.get("lich_su_gia", {})
    for ma_cophieu, records in lich_su_gia.items():
        # Kiểm tra từng bản ghi trong khoảng thời gian
        for record in records:
            date_str = record["date"]  # Giả sử trường "date" chứa ngày theo định dạng "dd/mm/yyyy"
            record_date = datetime.strptime(date_str, "%d/%m/%Y")  # Chuyển đổi chuỗi thành đối tượng datetime

            if start_date <= record_date <= end_date:
                ma_cophieu_with_data.append(ma_cophieu)
                break  # Thoát khỏi vòng lặp khi tìm thấy dữ liệu cho mã cổ phiếu này

# In ra tên các mã có dữ liệu
print("Các mã có dữ liệu từ {} đến {}:".format(start_date.strftime("%d/%m/%Y"), end_date.strftime("%d/%m/%Y")))
print(set(ma_cophieu_with_data))  # Sử dụng set để loại bỏ các mã trùng lặp


Các mã có dữ liệu từ 02/10/2024 đến 02/10/2024:
{'STB', 'ACB', 'TCB', 'LPB', 'ABB', 'EIB', 'NAB', 'KLB', 'BID', 'CTG', 'MBB', 'VIB', 'BVB', 'OCB', 'NVB', 'HDB', 'MSB', 'BAB', 'EVF', 'TPB', 'VAB', 'VCB', 'PGB', 'SSB', 'SHB', 'TIN', 'VPB', 'VBB', 'SGB'}


In [116]:
#13 in ra dữ liệu của 3 mã bất kì
# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  # Thay bằng tên chính xác của collection
collection = db[collection_name]

# Chỉ định ba mã cổ phiếu mà bạn muốn tìm
ma_cophieu_list = ["VCB", "BID", "CTG"]  # Thay thế bằng mã cổ phiếu bạn muốn tìm

# Lấy lịch sử giá cho từng mã cổ phiếu
for ma_cophieu in ma_cophieu_list:
    lich_su_gia = collection.find_one({"linhvuc": collection_name}, {"lich_su_gia": 1})

    if lich_su_gia and ma_cophieu in lich_su_gia["lich_su_gia"]:
        records = lich_su_gia["lich_su_gia"][ma_cophieu]
        
        print(f"Dữ liệu cho mã cổ phiếu {ma_cophieu}:")
        for record in records:
            print(record)
        print("\n")  # Thêm một dòng trống giữa các mã cổ phiếu
    else:
        print(f"Không tìm thấy dữ liệu cho mã cổ phiếu {ma_cophieu}.")


Dữ liệu cho mã cổ phiếu VCB:
{'date': '29/10/2024', 'opening_price': '92,100', 'highest_price': '92,600', 'lowest_price': '92,000', 'closing_price': '92,000', 'price_change': '-', 'percent_change': '-', 'volume': '1,297,600'}
{'date': '28/10/2024', 'opening_price': '91,500', 'highest_price': '92,300', 'lowest_price': '91,500', 'closing_price': '92,000', 'price_change': '+200', 'percent_change': '0.22%', 'volume': '1,105,800'}
{'date': '25/10/2024', 'opening_price': '92,100', 'highest_price': '92,300', 'lowest_price': '91,700', 'closing_price': '91,800', 'price_change': '+100', 'percent_change': '0.11%', 'volume': '996,300'}
{'date': '24/10/2024', 'opening_price': '92,200', 'highest_price': '92,300', 'lowest_price': '91,400', 'closing_price': '91,700', 'price_change': '+200', 'percent_change': '0.22%', 'volume': '755,400'}
{'date': '23/10/2024', 'opening_price': '91,200', 'highest_price': '91,800', 'lowest_price': '91,200', 'closing_price': '91,500', 'price_change': '+100', 'percent_cha

In [117]:
#14 Tính giá trung bình đóng cửa của một mã cổ phiếu trong khoảng thời gian nhất định

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng" 
collection = db[collection_name]

# Chỉ định mã cổ phiếu và khoảng thời gian
ma_cophieu = "VCB"  # Thay thế bằng mã cổ phiếu bạn muốn tính
start_date_str = "01/10/2024"  # Ngày bắt đầu
end_date_str = "31/10/2024"    # Ngày kết thúc

# Chuyển đổi ngày sang định dạng datetime
start_date = datetime.strptime(start_date_str, "%d/%m/%Y")
end_date = datetime.strptime(end_date_str, "%d/%m/%Y")

# Lấy lịch sử giá cho mã cổ phiếu
lich_su_gia = collection.find_one({"linhvuc": collection_name}, {"lich_su_gia": 1})

if lich_su_gia and ma_cophieu in lich_su_gia["lich_su_gia"]:
    records = lich_su_gia["lich_su_gia"][ma_cophieu]
    total_closing_price = 0
    count = 0

    # Tính giá trung bình đóng cửa
    for record in records:
        record_date = datetime.strptime(record["date"], "%d/%m/%Y")
        if start_date <= record_date <= end_date:
            closing_price = float(record["closing_price"].replace(",", ""))
            total_closing_price += closing_price
            count += 1

    if count > 0:
        average_closing_price = total_closing_price / count
        print(f"Giá trung bình đóng cửa của mã {ma_cophieu} từ {start_date_str} đến {end_date_str} là: {average_closing_price:.2f}")
    else:
        print(f"Không có dữ liệu cho mã {ma_cophieu} trong khoảng thời gian đã chỉ định.")
else:
    print(f"Không tìm thấy dữ liệu cho mã cổ phiếu {ma_cophieu}.")


Giá trung bình đóng cửa của mã VCB từ 01/10/2024 đến 31/10/2024 là: 91847.62


In [124]:
#15 Tìm mã cổ phiếu có sự thay đổi giá cao nhất trong một khoảng thời gian

# Kết nối tới MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['simplize']

# Tên collection chứa dữ liệu tài chính
collection_name = "Tài chính ngân hàng"  # Thay bằng tên chính xác của collection
collection = db[collection_name]

# Chỉ định khoảng thời gian
start_date_str = "01/10/2024"  # Ngày bắt đầu
end_date_str = "31/10/2024"    # Ngày kết thúc

# Chuyển đổi ngày sang định dạng datetime
start_date = datetime.strptime(start_date_str, "%d/%m/%Y")
end_date = datetime.strptime(end_date_str, "%d/%m/%Y")

# Tìm mã cổ phiếu có sự thay đổi giá cao nhất
max_price_change = float('-inf')
max_ma_cophieu = None

# Lấy tất cả các mã cổ phiếu trong collection
linhvuc_data = collection.find({"linhvuc": collection_name})

for linhvuc in linhvuc_data:
    for ma_cophieu, records in linhvuc["lich_su_gia"].items():
        for record in records:
            record_date = datetime.strptime(record["date"], "%d/%m/%Y")
            if start_date <= record_date <= end_date:
                percent_change_str = record["percent_change"].replace(",", "").replace("%", "")
                if percent_change_str and percent_change_str != '-':  # Kiểm tra giá trị
                    percent_change = float(percent_change_str)
                    if percent_change > max_price_change:
                        max_price_change = percent_change
                        max_ma_cophieu = ma_cophieu

if max_ma_cophieu:
    print(f"Mã cổ phiếu có sự thay đổi giá cao nhất từ {start_date_str} đến {end_date_str} là: {max_ma_cophieu} với sự thay đổi là: {max_price_change:.2f}%")
else:
    print("Không tìm thấy dữ liệu trong khoảng thời gian đã chỉ định.")



Mã cổ phiếu có sự thay đổi giá cao nhất từ 01/10/2024 đến 31/10/2024 là: EIB với sự thay đổi là: 6.94%
